# Chapter 3: Expectation, Variance and Cauchy–Schwarz

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch03/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_03.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Verifying the Cauchy–Schwarz Inequality

*Source file:* `code/ch03/c03_1_cauchy_schwarz_check.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
cov = [[4.0, 0.85 * 2.0 * 3.0], [0.85 * 2.0 * 3.0, 9.0]]
samples = rng.multivariate_normal([2.0, -1.0], cov, size=500_000)
X, Y = samples[:, 0], samples[:, 1]
print(f"(E[XY])^2: {(np.mean(X*Y))**2:.4f} <= E[X^2]E[Y^2]: {np.mean(X**2)*np.mean(Y**2):.4f}")

# Equality case: Y = a + bX  =>  |Corr| = 1 ; equality for the centred variables
Y_lin = 3.0 - 2.0 * X
Xc, Yc = X - X.mean(), Y_lin - Y_lin.mean()
print(f"Linear relation: (E[XcYc])^2 = {np.mean(Xc*Yc)**2:.4f} , E[Xc^2]E[Yc^2] = {np.mean(Xc**2)*np.mean(Yc**2):.4f}")
print(f"Sample correlation (for rho=0.85): {np.corrcoef(X, Y)[0,1]:.4f}")

(E[XY])^2: 9.5445 <= E[X^2]E[Y^2]: 79.7080
Linear relation: (E[XcYc])^2 = 63.7241 , E[Xc^2]E[Yc^2] = 63.7241
Sample correlation (for rho=0.85): 0.8489


## Uncorrelatedness versus Independence; the Law of Total Variance

*Source file:* `code/ch03/c03_2_uncorrelated_vs_independent_total_variance.py`

In [3]:
import numpy as np

rng = np.random.default_rng(42)
X = rng.normal(size=1_000_000)
Y = X**2
print(f"Corr(X, X^2) = {np.corrcoef(X, Y)[0,1]:+.4f}   (uncorrelated)")
print(f"P(Y>1 | |X|>1) = {np.mean(Y[np.abs(X) > 1] > 1):.1f}  vs  P(Y>1) = {np.mean(Y > 1):.4f}  (dependent)")

# Poisson-Gamma mixture: Var N = E[Var(N|L)] + Var(E[N|L])
alpha, beta = 3.0, 0.5
L = rng.gamma(alpha, 1/beta, size=1_000_000)   # numpy takes scale = 1/rate
N = rng.poisson(L)
print(f"E N   : MC={N.mean():.3f}  theory={alpha/beta:.3f}")
print(f"Var N : MC={N.var():.3f}  theory={alpha/beta + alpha/beta**2:.3f}  (would be {alpha/beta:.1f} for a Poisson)")

Corr(X, X^2) = -0.0006   (uncorrelated)
P(Y>1 | |X|>1) = 1.0  vs  P(Y>1) = 0.3171  (dependent)
E N   : MC=6.005  theory=6.000
Var N : MC=18.034  theory=18.000  (would be 6.0 for a Poisson)


## Jensen's Inequality: $\E[1/X]$ and $\E[S]$

*Source file:* `code/ch03/c03_3_jensen_inverse_and_s.py`

In [4]:
import numpy as np
from scipy.special import gammaln

rng = np.random.default_rng(42)
X = rng.gamma(5.0, 1.0, size=1_000_000)       # E X = 5
print(f"E[1/X] = {np.mean(1/X):.4f} >= 1/E[X] = {1/X.mean():.4f}  (theory E[1/X]=1/4)")

n, sigma = 5, 2.0
S = np.std(rng.normal(0, sigma, size=(500_000, n)), axis=1, ddof=1)
c4 = np.sqrt(2/(n-1)) * np.exp(gammaln(n/2) - gammaln((n-1)/2))
print(f"E[S] MC = {S.mean():.4f} < sigma = {sigma}  | theory c4*sigma = {c4*sigma:.4f}")

E[1/X] = 0.2501 >= 1/E[X] = 0.2000  (theory E[1/X]=1/4)
E[S] MC = 1.8787 < sigma = 2.0  | theory c4*sigma = 1.8800
